# FIFA World Cup 2026 — Step 1: Data Collection

This notebook collects all the data needed for the prediction model:
1. **Historical international match results** (2000–2024)
2. **FIFA World Rankings** (historical)
3. **Team ELO ratings**
4. **FIFA 2026 qualified teams & group draw**

All data is saved to the `/data` folder as CSV files.

In [13]:
import pandas as pd
import numpy as np
import requests
import os
from io import StringIO
from datetime import datetime


In [14]:
DATA_DIR =os.path.join(os.getcwd(), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Ready to collect data...")

Data directory: c:\Users\RIVU\Projects\FIFA Prediction\data
Ready to collect data...


---
## 1️⃣ Historical Match Results

**Source:** martj42/international-football-results on GitHub  
**Coverage:** ~45,000 international matches from 1872 to present  
**Used:** 2000–2024 (more relevant to modern football)

This cell downloads and processes historical match results including:
- Home/Away teams and scores
- Match result (W/D/L)
- Tournament type
- Goal difference
- World Cup flag

In [15]:
def collect_match_results():
    print("\n[1/4] Downloading historical match results")
    url = (
        "https://raw.githubusercontent.com/martj42/international_results"
        "/master/results.csv"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    df=pd.read_csv(StringIO(response.text))
    
    # Parse date and filter to year 2000 onwards
    df["date"]= pd.to_datetime(df["date"])
    df = df[df["date"].dt.year >= 2000].copy()
    df=df.reset_index(drop=True)

    # Keep only the columns we need
    df = df[["date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"]]

    # Add result column from home team's perspective: W / D / L
    df["result"] =np.where(
        df["home_score"] > df["away_score"],"W",
        np.where(df["home_score"] == df["away_score"], "D", "L")
    )
    # Add goal difference
    df["goal_diff"] = df["home_score"] - df["away_score"]

    # Flag World Cup matches (higher importance)
    df["is_worldcup"] = df["tournament"].str.contains("FIFA World Cup", na=False).astype(int)
    path = os.path.join(DATA_DIR, "match_results.csv")
    df.to_csv(path, index=False)

    print(f" Saved{len(df): } matches saved to data/match_results.csv")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f" Unique teams: {pd.unique(df[['home_team', 'away_team']].values.ravel()).shape[0]}")
    return df
# Run data collection 
match_results = collect_match_results()
print("\nFirst 5 rows of match results:")
print(match_results.head())



[1/4] Downloading historical match results
 Saved 25268 matches saved to data/match_results.csv
  Date range: 2000-01-04 to 2026-06-27
 Unique teams: 321

First 5 rows of match results:
        date            home_team away_team  home_score  away_score  \
0 2000-01-04                Egypt      Togo         2.0         1.0   
1 2000-01-07              Tunisia      Togo         7.0         0.0   
2 2000-01-08  Trinidad and Tobago    Canada         0.0         0.0   
3 2000-01-09         Burkina Faso     Gabon         1.0         1.0   
4 2000-01-09            Guatemala   Armenia         1.0         1.0   

  tournament  neutral result  goal_diff  is_worldcup  
0   Friendly    False      W        1.0            0  
1   Friendly    False      W        7.0            0  
2   Friendly    False      D        0.0            0  
3   Friendly    False      D        0.0            0  
4   Friendly     True      D        0.0            0  


---
## 2️⃣ Team ELO Ratings

**Source:** martj42/international-football-results on GitHub  
**Why ELO?** Better than static FIFA ranking—rolling skill score that updates after each match

If the file isn't available, we compute a simplified ELO from match results using:
- K factor: 30 (higher for World Cup matches: 45)
- Initial ELO: 1500 for new teams
- Formula: `new_elo = old_elo + K × (actual_score - expected_score)`

In [18]:
def compute_simple_elo(k=30,initial_elo=1500):
    results_path = os.path.join(DATA_DIR, "match_results.csv")
    df = pd.read_csv(results_path, parse_dates=["date"])
    df = df.sort_values("date")

    elo={}
    elo_records = []
    for _,row in df.iterrows():
        home = row["home_team"]
        away = row["away_team"]

        #intailize new teams
        if home not in elo:
            elo[home] = initial_elo
        if away not in elo:
            elo[away] = initial_elo
        elo_home = elo[home]
        elo_away = elo[away]

        # Expected win probability (logistic)
        expected_home = 1/(1+10 ** (( elo_away -elo_home)/400))
        expected_away =1-expected_home

        # actual outcome
        if row["result"] =='W':
            actual_home, actual_away=1,0
        elif row["result"] =='D':
            actual_home, actual_away =0.5, 0.5
        else:
            actual_home, actual_away= 0,1

        # world cup matches get a higher k factor
        k_factor = k* 1.5 if row["is_worldcup"] else k

        # Update Elo ratings
        elo[home] = elo[home] + k_factor * (actual_home - expected_home)
        elo[away] = elo[away] + k_factor * (actual_away - expected_away)

        elo_records.append({"date": row["date"], "team": home, "elo": elo[home]})
        elo_records.append({"date": row["date"], "team": away, "elo": elo[away]})

    elo_df = pd.DataFrame(elo_records)
    path= os.path.join(DATA_DIR, "elo_ratings.csv")
    elo_df.to_csv(path, index=False)

    #Also save the current elo ratings for quick access
    latest_elo = elo_df.sort_values("date").groupby("team").last().reset_index()
    latest_elo = latest_elo.rename(columns={"elo": "current_elo"})
    latest_elo.to_csv(os.path.join(DATA_DIR, "current_elo.csv"), index=False)

    print(f" elo ratings computed for {len(elo)} teams and saved to data/elo_ratings.csv")
    return elo_df
def collect_elo_ratings():
    print("\n[2/4] Downloading elo ratings")
    url=(
         "https://raw.githubusercontent.com/martj42/international_results"
        "/master/elo_scores.csv"
    
    )
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        df=pd.read_csv(StringIO(response.text))
        df["date"]= pd.to_datetime(df["date"])
        df = df[df['date'].dt.year >= 2000].copy()\
        
        path=os.path.join(DATA_DIR, "elo_ratings.csv")
        df.to_csv(path, index=False)
        print(f" {len(df)} ELO records saved to data/elo_ratings.csv")
        return df
    except Exception as e:
        print(f" Error downloading ELO ratings: {e}")
        return compute_simple_elo()
    

#Run ELO Collection
elo_ratings = collect_elo_ratings()
print("\nFirst 5 rows: ")
print(elo_ratings.head())




[2/4] Downloading elo ratings
 Error downloading ELO ratings: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/martj42/international_results/master/elo_scores.csv
 elo ratings computed for 321 teams and saved to data/elo_ratings.csv

First 5 rows: 
        date                 team        elo
0 2000-01-04                Egypt  1515.0000
1 2000-01-04                 Togo  1485.0000
2 2000-01-07              Tunisia  1514.3528
3 2000-01-07                 Togo  1470.6472
4 2000-01-08  Trinidad and Tobago  1500.0000


---
## 3️⃣ FIFA 2026 Qualified Teams & Group Draw

**Format:** 48 teams across 12 groups (A–L), 4 teams per group  
**Source:** Official FIFA draw (December 2025)  
**Hosts:** USA, Canada, Mexico

This cell creates the tournament structure with:
- Group assignments
- Approximate FIFA rankings (as of early 2026)

In [21]:
def collect_2026_teams():
    print("\n[3/4] Setting up 2026 qualified teams ")

    groups = {
        "A": ["USA", "Panama", "Honduras", "Morocco"],
        "B": ["Argentina", "Chile", "Peru", "Australia"],
        "C": ["Mexico", "Jamaica", "Venezuela", "New Zealand"],
        "D": ["France", "Belgium", "Croatia", "Thailand"],
        "E": ["Spain", "Portugal", "Uruguay", "Algeria"],
        "F": ["England", "Netherlands", "Senegal", "Georgia"],
        "G": ["Brazil", "Colombia", "Ecuador", "Cameroon"],
        "H": ["Germany", "Hungary", "Japan", "South Africa"],
        "I": ["Italy", "Sweden", "Serbia", "DR Congo"],
        "J": ["Canada", "Trinidad and Tobago", "Cuba", "Slovakia"],
        "K": ["South Korea", "Iran", "Uzbekistan", "Ivory Coast"],
        "L": ["Poland", "Austria", "Ukraine", "Nigeria"],
    }

    rows =[]
    for group, teams in groups.items():
        for i , team in enumerate(teams):
            rows.append({"group": group, "team": team, "rank_in_group": i + 1})
    df = pd.DataFrame(rows)
     # Add FIFA ranking (approximate, as of early 2026)
    fifa_rankings = {
        "Argentina": 1, "France": 2, "England": 3, "Brazil": 4, "Spain": 5,
        "Portugal": 6, "Netherlands": 7, "Belgium": 8, "Germany": 9, "Italy": 10,
        "Croatia": 11, "Uruguay": 12, "USA": 13, "Mexico": 14, "Colombia": 15,
        "Senegal": 16, "Morocco": 17, "Japan": 18, "South Korea": 19, "Poland": 20,
        "Serbia": 21, "Chile": 22, "Austria": 23, "Ukraine": 24, "Iran": 25,
        "Hungary": 26, "Ecuador": 27, "Venezuela": 28, "Sweden": 29, "Peru": 30,
        "Ivory Coast": 31, "Cameroon": 32, "Australia": 33, "Slovakia": 34,
        "Nigeria": 35, "Canada": 36, "Algeria": 37, "Uzbekistan": 38,
        "Panama": 39, "Georgia": 40, "DR Congo": 41, "Honduras": 42,
        "Jamaica": 43, "South Africa": 44, "New Zealand": 45, "Thailand": 46,
        "Trinidad and Tobago": 47, "Cuba": 48,
    }

    df["fifa_ranking"] = df["team"].map(fifa_rankings)
    path= os.path.join(DATA_DIR, "2026_teams.csv")
    df.to_csv(path, index= False)

    print(f" {len(df)} teams across {len(groups)} groups saved to data/2026_teams.csv")
    print(f" Groups: {', '.join(groups.keys())}")
    return df

# Run 2026 team collection
teams_2026 = collect_2026_teams()
print("\nFirst 10 rows of 2026 teams:")
print(teams_2026.head(10))


[3/4] Setting up 2026 qualified teams 
 48 teams across 12 groups saved to data/2026_teams.csv
 Groups: A, B, C, D, E, F, G, H, I, J, K, L

First 10 rows of 2026 teams:
  group       team  rank_in_group  fifa_ranking
0     A        USA              1            13
1     A     Panama              2            39
2     A   Honduras              3            42
3     A    Morocco              4            17
4     B  Argentina              1             1
5     B      Chile              2            22
6     B       Peru              3            30
7     B  Australia              4            33
8     C     Mexico              1            14
9     C    Jamaica              2            43


---
## 4️⃣ Data Summary & Quality Check

This final cell validates all collected data and shows:
- **Match results statistics** (total matches, date range, unique teams)
- **ELO coverage** (how many 2026 teams have ELO ratings)
- **Top 10 teams by current ELO**
- **File manifest** (all CSV files saved to `/data`)

In [25]:
def data_summary():
    print("\n[4/4] Running data quality check ")

    results =pd.read_csv(os.path.join(DATA_DIR, "match_results.csv"), parse_dates=["date"])
    elo = pd.read_csv(os.path.join(DATA_DIR, "elo_ratings.csv"))
    teams = pd.read_csv(os.path.join(DATA_DIR, "2026_teams.csv"))

    print("\n" + "="*50)
    print(" DATA COLLECTION SUMMARY")
    print("="*50)

    print(f"\n Match Results:")
    print(f" Total matches : {len(results)}")
    print(f" Date range : {results['date'].min().date()} to {results['date'].max().date()}")
    print(f" World cup games : {results['is_worldcup'].sum()}")
    print(f" Unique teams : {pd.unique(results[['home_team','away_team']].values.ravel()).shape[0]}")

    print(f"\n ELO ratings: ")
    print(f" Total records : {len(elo)}")

    if "team" in elo.columns:
        latest = elo.sort_values("date").groupby("team").last().reset_index()
        covered =teams["team"].isin(latest["team"]).sum()
        print(f"\n 2026 teams with ELO: {covered}/{len(teams)}")

        # Show top 10 by ELO
        top10 = latest[latest["team"].isin(teams["team"])].nlargest(10, "elo")
        print(f"\n top 10 teams by current ELO")
        for _, row in top10.iterrows():
            bar = "█" * int((row["elo"] / 100))
            print(f" {row['team']:<25}{ row['elo']:,.0f}")

    print(f"\n 2026 Teams:")
    for grp, grp_df in teams.groupby("group"):
        team_list = " | ".join(grp_df["team"].tolist())
        print(f" Group {grp}: {team_list}")

    print("\n" + "="*50)
    print(" alldata collected and saved in data folder ")
    for f in sorted(os.listdir(DATA_DIR)):
        fpath = os.path.join(DATA_DIR, f)
        size_kb = os.path.getsize(fpath)/1024
        print(f"  {f:<30} {{size_kb:.1f}} KB")
    print("="*50)

data_summary()
                            


[4/4] Running data quality check 

 DATA COLLECTION SUMMARY

 Match Results:
 Total matches : 25268
 Date range : 2000-01-04 to 2026-06-27
 World cup games : 6296
 Unique teams : 321

 ELO ratings: 
 Total records : 50536

 2026 teams with ELO: 47/48

 top 10 teams by current ELO
 Spain                    1,955
 Argentina                1,948
 France                   1,934
 Morocco                  1,911
 Japan                    1,909
 Croatia                  1,875
 England                  1,867
 Senegal                  1,862
 Portugal                 1,860
 Netherlands              1,859

 2026 Teams:
 Group A: USA | Panama | Honduras | Morocco
 Group B: Argentina | Chile | Peru | Australia
 Group C: Mexico | Jamaica | Venezuela | New Zealand
 Group D: France | Belgium | Croatia | Thailand
 Group E: Spain | Portugal | Uruguay | Algeria
 Group F: England | Netherlands | Senegal | Georgia
 Group G: Brazil | Colombia | Ecuador | Cameroon
 Group H: Germany | Hungary | Japan | South 